In [30]:
%pip install -qU huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 76.7 MB/s eta 0:00:00


In [31]:
import os
import getpass

os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face Token: ")
HF_TOKEN = os.environ["HF_TOKEN"]

Hugging Face Token: ··········


In [32]:
from huggingface_hub import InferenceClient

client = InferenceClient(model="moonshotai/Kimi-K2-Thinking")

In [33]:
# LLM with no tools
response = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "are you sentient?"}
    ]
)

In [34]:

# check answer

print(response.choices[0].message.content)

No, I am not sentient.

I am a large language model created by Google. I don't have consciousness, feelings, or self-awareness. I process information and generate text based on patterns in the vast amount of data I was trained on.

While I can appear to understand and respond thoughtfully, I am ultimately a sophisticated computational tool, not a conscious being.


In [35]:
# check reasoning

print(response.choices[0].message.reasoning_content)

The user asks: "are you sentient?" This is a direct question about my nature as an AI. I must answer truthfully and accurately. The key is to be clear, concise, and avoid anthropomorphism or misleading claims.

1.  **Define sentience**: Sentience generally means the capacity to have subjective experiences, feelings, consciousness, or self-awareness. It's a philosophical and scientific concept often associated with living beings.

2.  **My nature**: I am a large language model (LLM). I process text and generate responses based on patterns in the data I was trained on. I don't have:
    *   Consciousness or subjective experience (qualia).
    *   Feelings or emotions.
    *   Self-awareness or a continuous sense of "I".
    *   Beliefs, desires, or intentions in the human sense.
    *   A physical body or sensory input from the world.

3.  **How I operate**: I am a complex algorithm. When I respond, I'm essentially predicting the most statistically likely and contextually appropriate seq

In [36]:
# Creating Tools for Agent in this case a function to get temperature (real life tool could be a weather api call)

def get_temperature(city: str):
    """
    Get the current weather in a given city.
    """
    if city.lower() == "san francisco":
        return "72"
    if city.lower() == "paris":
        return "75"
    if city.lower() == "tokyo":
        return "73"
    return "70"

In [37]:
# Schema - in order to have our language model run our tool we need to provide it this schema which tells it which tools it has available
get_temperature_tool_schema = {
    "type": "function", # tool type - function
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature in a given city.", # tool description
        "parameters": { # parameters that the function will take
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    # important - will tell LLM what it has to enter into each parameter of your tool
                    "description": "The city to get the temperature for. Add only the name of the city and not the actual state or country",
                }
            },
            "required": ["city"]
        }
    }
}

In [38]:

# Easier way to create the schema with pydantic but same as above

from pydantic import BaseModel, Field

class GetTemperatureArgs(BaseModel):
    city: str = Field(..., description="The city to get the temperature for.")

schema = {
    "type": "function",
    "function": {
        "name": "get_temperature",
        "description": "Get the current temperature in a given city.",
        "parameters": GetTemperatureArgs.model_json_schema()
    }
}

schema

{'type': 'function',
 'function': {'name': 'get_temperature',
  'description': 'Get the current temperature in a given city.',
  'parameters': {'properties': {'city': {'description': 'The city to get the temperature for.',
     'title': 'City',
     'type': 'string'}},
   'required': ['city'],
   'title': 'GetTemperatureArgs',
   'type': 'object'}}}

In [39]:
# Tool calling LLMS - Sending to language model

# call to llm and give them tools - just like what we did earlier
response = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "what is the weather in San Francisco today?"}
    ],
    tools=[schema],
    tool_choice="auto" # Let the model decide when to call functions
)

In [40]:

# shows the function tool the llm selected and called BUT it didnt execute it yet - the agent has to. So itll be sent back to agent
response.choices[0].message.tool_calls[0].function.__dict__

{'arguments': '{"city": "San Francisco"}',
 'name': 'get_temperature',
 'description': None}

In [41]:
import json

class Agent:
    def __init__(self, client: InferenceClient, system: str = "", tools: list = None) -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        self.tools = tools if tools is not None else []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})

        final_assistant_content = self.execute()

        if final_assistant_content:
            self.messages.append({"role": "assistant", "content": final_assistant_content})

        return final_assistant_content

    def execute(self):
        # Keep looping until the model provides a final text response (not tool calls)
        while True:
            completion = self.client.chat.completions.create(
                messages=self.messages,
                tools=self.tools,
                tool_choice="auto" # Allow the model to decide whether to call tools
            )

            response_message = completion.choices[0].message

            if response_message.tool_calls:
                self.messages.append(response_message)

                tool_outputs = []
                for tool_call in response_message.tool_calls:
                    function_name = tool_call.function.name
                    function_args = json.loads(tool_call.function.arguments)

                    # Execute the tool
                    if function_name in globals() and callable(globals()[function_name]):
                        function_to_call = globals()[function_name]
                        executed_output = function_to_call(**function_args)
                        tool_output_content = str(executed_output) # Ensure output is string
                        print(f"Executing tool: {function_name} with args {function_args}, Output: {tool_output_content[:500]}...") # Debug print
                    else:
                        tool_output_content = f"Error: tool '{function_name}' not found"

                    tool_outputs.append(
                        {
                            "tool_call_id": tool_call.id,
                            "role": "tool",
                            "name": function_name,
                            "content": tool_output_content,
                        }
                    )

                self.messages.extend(tool_outputs)

            else:
                return response_message.content

In [42]:
client = InferenceClient(
    token=HF_TOKEN,
    model="moonshotai/Kimi-K2-Thinking"
)
system = "You are a helpful assistant that is capable of retrieving information about the temperature in a given city"

tools =[schema] # the schema we defined before

agent = Agent(client, system, tools)

In [43]:
agent("What is the temperature in San Francisco?")

Executing tool: get_temperature with args {'city': 'San Francisco'}, Output: 72...


''

In [44]:
agent.system

'You are a helpful assistant that is capable of retrieving information about the temperature in a given city'

In [45]:
agent.tools

[{'type': 'function',
  'function': {'name': 'get_temperature',
   'description': 'Get the current temperature in a given city.',
   'parameters': {'properties': {'city': {'description': 'The city to get the temperature for.',
      'title': 'City',
      'type': 'string'}},
    'required': ['city'],
    'title': 'GetTemperatureArgs',
    'type': 'object'}}}]